In [ ]:
# DR baseline v4: as v3 (APTOS + DDR, strong augmentation) but the outer 10% of each photo's radius is hidden
# in training and testing (--fov-frac 0.9), to remove the camera/hospital shortcut found by the v3 rim test.
import glob, os, tarfile
import pandas as pd
_ = [tarfile.open(b, "r:xz").extractall("/kaggle/working") for b in sorted(glob.glob("/kaggle/input/**/dr_*_code.bin", recursive=True))]
os.chdir("/kaggle/working/dr-baseline")
APTOS_DIR = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train.csv", recursive=True) if os.path.isdir(os.path.join(os.path.dirname(p), "train_images"))][0]
DDR_DIR = os.path.dirname(glob.glob("/kaggle/input/**/DR_grading", recursive=True)[0])
GT_TRAIN = glob.glob("/kaggle/input/**/2. Groundtruths/a.*.csv", recursive=True)[0]
GT_TEST = glob.glob("/kaggle/input/**/2. Groundtruths/b.*.csv", recursive=True)[0]
IMG_TRAIN = glob.glob("/kaggle/input/**/1. Original Images/a. Training Set", recursive=True)[0]
IMG_TEST = glob.glob("/kaggle/input/**/1. Original Images/b. Testing Set", recursive=True)[0]
for part, gt in (("train", GT_TRAIN), ("test", GT_TEST)):
    lab = pd.read_csv(gt).iloc[:, :3]
    lab.columns = ["Image name", "Retinopathy grade", "Risk of macular edema"]
    lab.dropna().to_csv(f"/kaggle/working/idrid_{part}.csv", index=False)
print("APTOS:", APTOS_DIR, "| DDR:", DDR_DIR, "| files:", sorted(os.listdir(".")))

In [ ]:
!python prepare_data.py train-sets --aptos-dir {APTOS_DIR} --ddr-dir {DDR_DIR} --out-dir data/processed

In [ ]:
# Diagnostic: do DDR file names encode a hospital, and do DDR's validation and test folders differ?
m = pd.read_csv("data/processed/manifest.csv"); d = m[m["source"] == "ddr"].copy()
print("example DDR ids:", d["image_id"].head(5).tolist())
d["prefix"] = d["image_id"].astype(str).str.split("[-_]", regex=True).str[0]
d["referable"] = d["grade"] >= 2
print(d["prefix"].nunique(), "distinct name prefixes")
if d["prefix"].nunique() < len(d) / 5:
    tab = d.groupby(["prefix", "split"]).size().unstack(fill_value=0)
    print(tab.sort_values("train", ascending=False).head(12).to_string())
    only_test = sorted(set(d.loc[d["split"] == "test", "prefix"]) - set(d.loc[d["split"] != "test", "prefix"]))
    print("prefixes only in test:", len(only_test), only_test[:15])
    rate = d.groupby("prefix")["referable"].agg(["mean", "size"]).query("size >= 30")
    print(f"referable rate by prefix (>=30 photos): min {rate['mean'].min():.2f}, median {rate['mean'].median():.2f}, max {rate['mean'].max():.2f} over {len(rate)} prefixes")

In [ ]:
!python train.py --manifest data/processed/manifest.csv --out runs/v4 --aug strong --epochs 8 --fov-frac 0.9

In [ ]:
!python evaluate.py --run-dir runs/v4 --csv data/processed/manifest.csv --split test

In [ ]:
!python prepare_data.py external --name idrid_train --csv /kaggle/working/idrid_train.csv --image-dir "{IMG_TRAIN}" --image-col "Image name" --label-col "Retinopathy grade" --dme-col "Risk of macular edema" --out-dir data/processed
!python prepare_data.py external --name idrid_test --csv /kaggle/working/idrid_test.csv --image-dir "{IMG_TEST}" --image-col "Image name" --label-col "Retinopathy grade" --dme-col "Risk of macular edema" --out-dir data/processed
pd.concat([pd.read_csv("data/processed/external_idrid_train.csv"), pd.read_csv("data/processed/external_idrid_test.csv")]).to_csv("data/processed/external_idrid_all.csv", index=False)
!python evaluate.py --run-dir runs/v4 --csv data/processed/external_idrid_all.csv

In [ ]:
# Site tuning, done fairly: threshold from IDRiD's training folder, measured on IDRiD's testing folder
!python evaluate.py --run-dir runs/v4 --csv data/processed/external_idrid_train.csv --n-boot 0
!python site_threshold.py --predictions runs/v4/eval_external_idrid_train/predictions.csv --thresholds runs/v4/thresholds.json --out runs/v4/thresholds_idrid.json
!python evaluate.py --run-dir runs/v4 --csv data/processed/external_idrid_test.csv --thresholds runs/v4/thresholds_idrid.json --name idrid_test_site_tuned

In [ ]:
!python rim_test.py --run-dir runs/v4 --csv data/processed/external_idrid_all.csv --split external --out runs/v4/rim_test_idrid

In [ ]:
!rm -rf data/processed/images runs/v4/last.pt; du -sh /kaggle/working; find /kaggle/working -type f | wc -l